# Initalisation

## Import libraries

In [ ]:

import os 

import brightway2 as bw
import lca_algebraic as agb
import numpy as np
from sympy import init_printing
import bw2io
import bw2data as bd
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import pandas as pd

# Pretty print for Sympy
init_printing()
bw.projects

: 

## Init brightway2 and databases

In [ ]:
# Set the current project
# Can be any name
bw.projects.set_current('Project_name')

# It's better to not leave credential in the code.
# Create a file named .env, that you will not share /commit, and contains the following :
ECOINVENT_LOGIN="login"
ECOINVENT_PASSWORD="password"

# This load .env file into os.environ
load_dotenv()

# This downloads ecoinvent and installs biopshere + technosphere + LCIA methods
if len(bw.databases) > 0:
    print("Initial setup already done, skipping")
   
else:
    # This is now the prefered method to init an Brightway2 with Ecoinvent
    # It is not more tied to a specific version of bw2io
    bw2io.import_ecoinvent_release(
        version="3.10",
        system_model="cutoff",
        username=ECOINVENT_LOGIN, # Read for .env file
        password=ECOINVENT_PASSWORD, # Read from .env file
        use_mp=True)
bw.databases


In [ ]:
# Define the name of your new databse
DB_project_baseline = 'Project_name'
# This is better to cleanup the whole foreground model each time, and redefine it in the notebook (or a python file)
# instead of relying on a state or previous run.
# Any persistent state is prone to errors.
agb.resetDb(DB_project_baseline)

# Parameters are stored at project level : 
# Reset them also
# You may remove this line if you import a project and parameters from an external source (see loadParam(..))
agb.resetParams()

# Overview of the databases
agb.list_databases()

## Define the impacts categories

In [ ]:
# below is all the impacts categories of LCIA methodology : EF v3.0

climat = agb.findMethods("climate change", mainCat="EF v3.0")
resources = agb.findMethods("material resources: metals/minerals", mainCat="EF v3.0")
energy = agb.findMethods("energy resources: non-renewable", mainCat="EF v3.0")
acid = agb.findMethods("acidification", mainCat="EF v3.0")
ecotox = agb.findMethods("ecotoxicity: freshwater", mainCat="EF v3.0")
carcinogenic = agb.findMethods("human toxicity: carcinogenic", mainCat="EF v3.0")
NoCarcinogenic = agb.findMethods("human toxicity: non-carcinogenic", mainCat="EF v3.0")
eutrophFreshwater = agb.findMethods("eutrophication: freshwater", mainCat="EF v3.0")
eutrophMarine = agb.findMethods("eutrophication: marine", mainCat="EF v3.0")
eutrophTerrestrial = agb.findMethods("eutrophication: terrestrial", mainCat="EF v3.0")
ionising = agb.findMethods("ionising radiation: human health", mainCat="EF v3.0")
landuse = agb.findMethods("land use", mainCat="EF v3.0")
ozone = agb.findMethods("ozone depletion ", mainCat="EF v3.0")
particulate = agb.findMethods("particulate matter formation", mainCat="EF v3.0")
oxidant = agb.findMethods("photochemical oxidant formation: human health", mainCat="EF v3.0")
water = agb.findMethods("water use", mainCat="EF v3.0")



all_impacts_categories = [climat[0], resources[0], energy[0], acid[0], ecotox[0], carcinogenic[0], NoCarcinogenic[0], eutrophFreshwater[0], eutrophMarine[0],
                 eutrophTerrestrial[0], ionising[0], landuse[0], ozone[0], particulate[0], oxidant[0], water[0]]

# Create baseline environmental impacts model

### Define control parameters for functional unit

1. Service provision time
2. Technical lifetime
3. Number of baseline products need to be manufactured during the service provision time

In [ ]:
# Y_service = agb.newFloatParam(
#     'Y_service', 
#     label="service provision time",
#     default=?, # Fixed if no min /max provided
#     distrib=agb.DistributionType.FIXED,
#     description="service provision years")

# Y_p = agb.newFloatParam(
#     'Y_p', 
#     label="technical lifetime of product",
#     default=?, # Fixed if no min /max provided
#     distrib=agb.DistributionType.FIXED,
#     description="technical lifetime of product")

# N_p_baseline = Y_service//Y_p
# print(N_p_baseline)


## Manufacturing phase modelling
To effectively allocate manufacturing impacts to the specific functions provided by the PEC, a data structure organized around the product's technical blocks is proposed.



### Define parameters related to technical blocks manufacturing

With data for the following parameters :
1. Unit impact of component manufacturing
2. Component's size
3. Location of manufacturing site


Example:

**Define parameters for technical block i**

In [ ]:
# Technical_block_i = agb.newActivity(DB_project_baseline, 
#         "Technical_block_i", 
#         "unit",
#        exchanges = {
#             agb.findActivity("component's reference in ecoinvent", loc="loc", db_name="ecoinvent-3.10-cutoff"): component's mass in kg, 
#             # mark if it is target component for "repair" or "reuse"
#                  }) 


**Start parameters definition**

In [ ]:
# start parameters definition!

### Compile the manufacturing impacts of baseline scenario

In [ ]:
# EI_manufacturing_baseline=agb.newActivity(DB_project_baseline, "EI_manufacturing_baseline", "unit", {
#     Technical_block_1:N_p_baseline,
#     Technical_block_2:N_p_baseline,
#     Technical_block_i:N_p_baseline,
# })
# agb.printAct(EI_manufacturing_baseline) 

### Calculate impacts

In [ ]:
# EI_manufacturing_baseline=agb.newActivity(DB_project_baseline, "EI_manufacturing_baseline", "unit", {
#     Technical_block_1:N_p_baseline,
#     Technical_block_2:N_p_baseline,
#     Technical_block_i:N_p_baseline,
# })
# agb.printAct(EI_manufacturing_baseline) 

## Transport phase modelling

## Define control parameters for transport phase

For transport i, define
1. Transport type
2. Transport distance
3. Transported mass

**Hypothesis (suggestion)** <br>
In the absence of specific data for the transport stage (distance and transport modes),default values from the Product Category Rules for Electrical and Electronic prod-ucts  were utilized.  

These default assumptions include 19,000 km by boat and 1,000 km by lorry for international transport, 3,500 km by lorry for intracontinentaltransport, and 1,000 km by lorry for local or domestic transport from the manufacturing stage to end-of-life treatment.

For the manufacturing stage, international transport is assumed to consist of 19,000km by boat and 1,000 km by lorry.  

For the end-of-life stage, based on the Frenche-waste data, it is assumed that 44.3% of the productis treated locally within the French e-waste management system, requiring 1,000 kmof transport by lorry. 
The remaining 55.7% of the product is assumed to be exportedabroad, involving 19,000 km by boat and an additional 1,000 km by lorry

In [ ]:
# EI_manufacturing_baseline=agb.newActivity(DB_project_baseline, "EI_manufacturing_baseline", "unit", {
#     Technical_block_1:N_p_baseline,
#     Technical_block_2:N_p_baseline,
#     Technical_block_i:N_p_baseline,
# })
# agb.printAct(EI_manufacturing_baseline) 

### Compile transport impacts of baseline scenario

In [ ]:

# EI_transport_baseline = agb.newActivity(DB_project_baseline, "EI_transport_baseline", "unit", {
# Transport_manufacturing_baseline:1,
# Transport_eol_baseline:1
#     })

# agb.printAct(EI_transport_baseline)


## Use phase modelling

### Define control parameters

For use stage modelling, below are the control parameters to be defined :
1. Average power losses (P_loss) )
2. Total operation time _op(Twer

In [ ]:
# # parameters definition

# T_op= agb.newFloatParam(
#     'T_op', 
#     label="total operation time",
#     default=?, # Fixed if no min /max provided
#     distrib=agb.DistributionType.FIXED,
#     description="total operation time")

# P_loss = agb.newFloatParam(
#     'P', 
#     label="Average power losses ",
#     default=?, # Fixed if no min /max provided
#     distrib=agb.DistributionType.FIXED,
#     description="Average power losses /kW")


### Compile use impacts of baseline scenario

In [ ]:
# EI_use_baseline=agb.newActivity(DB_project_baseline, "EI_use_baseline", "unit", {
#     agb.findActivity("market for electricity, low voltage", loc="FR", db_name="ecoinvent-3.10-cutoff"): P_loss*T_op,
#     })

# agb.printAct(EI_use_baseline)

## Repair phase modelling

### Define control parameters

Control parameters for repair stages are:
1. number of repair intervention
2. effective repair rate
3. target components to repair

In [ ]:
# RP= agb.newFloatParam(
#     'RP', 
#     label="number of repair intervention",
#     default=?,# Fixed if no min /max provided
#     distrib=agb.DistributionType.FIXED,
#     description="number of repair intervention")

# r_repair=agb.newFloatParam(
#     'r_repair', 
#     label="effective repair rate",
#     default=?,# Fixed if no min /max provided
#     distrib=agb.DistributionType.FIXED,
#     description="effective repair rate")


# EI_target_repair_components=agb.newActivity(DB_project_baseline, "EI_target_repair_components", "unit", 
#                                          exchanges={

#                                          })                                         

# df_EI_target_repair_components= agb.printAct(EI_target_repair_components).sum().to_frame().T
# Amount_kg_EI_target_repair_components = df_EI_target_repair_components.iloc[:, 1].tolist()
# M_repair_components=Amount_kg_EI_target_repair_components[0]

# Transport_repair_components =agb.newActivity(DB_project_baseline,
#         "Transport_repair_components", 
#         "unit",{

#         })


### Compile repair impacts of baseline scenario

In [ ]:
# EI_repair=agb.newActivity(DB_project_baseline, "EI_repair", "unit", 
#                                          exchanges={
#                 agb.findActivity("EI_target_repair_components",db_name=DB_project_baseline):r_repair*RP,
#                 agb.findActivity("EI_manufacturing_scenario",db_name=DB_project_baseline):((1-r_repair)*RP) 
#                                          })    

## reuse phase modelling

### Define control parameters

Control parameters for reuse phase are :
1. target component for reuse & its manufacturing and end-of-life impacts
1. target component's functional usure during the first service

In [ ]:
# a_component_i=agb.newFloatParam(
#     'a_component_i', 
#     label="functional usure during the first service of component i",
#     default=?,min=?,max=?, # Fixed if no min /max provided
#     distrib=agb.DistributionType.LINEAR,
#     description="functional usure during the first service of component i")


# EI_avoided_Target_reuse_components_manufacturing =agb.newActivity(DB_project_baseline,
#         "EI_avoided_Target_reuse_components_manufacturing", 
#         "unit",{
#             agb.findActivity("component i's reference", db_name="ecoinvent-3.10-cutoff"):size_of_component_i*(1-a_component_i),
#  })

# EI_avoided_Target_reuse_components_eol =agb.newActivity(DB_project_baseline,
#         "EI_avoided_Target_reuse_components_eol", 
#         "unit",{
#             agb.findActivity("eol_process", db_name="ecoinvent-3.10-cutoff"):size_of_component_i*(1-a_component_i),
#  })


### Compile reuse impacts of baseline scenario

In [ ]:

# EI_reuse =agb.newActivity(DB_project_baseline,
#         "EI_reuse", 
#         "unit",{
#                 agb.findActivity("EI_avoided_Target_reuse_components_manufacturing",db_name=DB_project_baseline):-(N_p_baseline+(1-r_repair)*RP),
#                 agb.findActivity("EI_avoided_Target_reuse_components_eol",db_name=DB_project_baseline):-(N_p_baseline+(1-r_repair)*RP)
#        })

## Recycling stage

Control parameters for reuse phase are :
1. target components for recycling,
2. quantity of the components,
3. recycling process.

In [ ]:
# Example of plastic recycling and copper recycling
# EI_eol_recycling_plastiques=agb.newActivity(DB_project_baseline, "EI_eol_recycling_plastiques", "unit", {
#       agb.findActivity("plastic flake, consumer electronics, for recycling, Recycled Content cut-off", loc="GLO", db_name="ecoinvent-3.10-cutoff"):??,
#    }
# )

# EI_eol_recycling_copper=agb.newActivity(DB_project_baseline, "EI_eol_recycling_copper", "unit", {
#       agb.findActivity("treatment of electronics scrap, metals recovery in copper smelter", loc="GLO", db_name="ecoinvent-3.10-cutoff"): ??,
#    }
# )

# EI_recycling =agb.newActivity(DB_project_baseline,
#         "EI_recycling", 
#         "unit",{
#                 agb.findActivity("EI_eol_recycling_plastiques",db_name=DB_project_baseline):1,
#                 agb.findActivity("EI_eol_recycling_copper",db_name=DB_project_baseline):1
#        })

## End-of-life phase modelling



### Define control parameters

End-of-life phase consist of two parts :
1. Formal disposal : reuse, recycling, energy recovery by incineration, incineration, landfill
2. Informal disposal: mainly uncontrolled incineration and landfill 

Control parameters
1. Mass of product
2. Formal end-of-use collection rate
3. Recycling rate & Unit impacts of recycling process
4. Energy recovery through incineration rate & Unit impacts of energy recovery process
5. Controlled/Uncontrolled Incineration rate & Unit impacts of controlled/Uncontrolled incineration process
6. Controlled/Uncontrolled Landfill rate & Unit impacts of controlled/Uncontrolled landfill process
7. reuse rate & Unit impacts of reuse process
8. Percentage of unformal treatment without developed WEEE management infrastructure


**Hypothesis (suggestions)**<br>
It is assumed that the formal end-of-use collection rate is 44.3%, within which the recycling rate is 75.2%, energy recovery through incineration accounts for 9.8%, and landfill and incineration without energy recovery are estimated to be equally split, each representing 6.65%. The whole product reuse rate is assumed to be 1.8%, with one reuse cycle.

The remaining 55.7% of waste is undocumented and considered to be informally disposed of. For this project, the assumption is made that 100% of informal treatment is exported abroad and processed in waste management systems lacking developed WEEE infrastructure, with 50\% incinerated and 50% sent to landfill.

In [ ]:

# #Parameters definition

# R_eou = agb.newFloatParam(
#     'R_eou', 
#     label="formal end-of-use collection rate",
#     default=?, # Fixed if no min /max provided
#     distrib=agb.DistributionType.FIXED,
#     description="formal end-of-use collection rate")

# R_recycling = agb.newFloatParam(
#     'R_recycling', 
#     label="recycling rate",
#     default=?, # Fixed if no min /max provided
#     distrib=agb.DistributionType.FIXED,
#     description="recycling rate")

# ER = agb.newFloatParam(
#     'ER', 
#     label="energy recovery through incineration rate ",
#     default=?, # Fixed if no min /max provided
#     distrib=agb.DistributionType.FIXED,
#     description="energy recovery through incineration ")

# INC = agb.newFloatParam(
#     'INC', 
#     label="incineration rate",
#     default=?, # Fixed if no min /max provided
#     distrib=agb.DistributionType.FIXED,
#     description="incineration")

# LF = agb.newFloatParam(
#     'LF', 
#     label="landfill rate",
#     default=?, # Fixed if no min /max provided
#     distrib=agb.DistributionType.FIXED,
#     description="landfill")

# R_reuse = agb.newFloatParam(
#     'R_reuse', 
#     label="reuse rate",
#     default=?, # Fixed if no min /max provided
#     distrib=agb.DistributionType.FIXED,
#     description="landfill")

# WINF = agb.newFloatParam(
#     'WINF', 
#     label="Percentage of unformal treatment without developed WEEE management infrastructure",
#     default=?, # Fixed if no min /max provided
#     distrib=agb.DistributionType.FIXED,
#     description="Percentage of unformal treatment without developed WEEE management infrastructure ")



# EI_eol_baseline_formal=agb.newActivity(DB_project_baseline, "EI_eol_baseline_formal", "unit", {
#       agb.findActivity("treatment of waste electric and electronic equipment, shredding", loc="GLO", db_name="ecoinvent-3.10-cutoff"): R_eou*R_recycling*M_p_baseline,
#     agb.findActivity("treatment of used capacitor, to hazardous waste incineration, with energy recovery", unit="kilogram", loc="RoW", db_name="ecoinvent-3.10-cutoff"): R_eou*ER*M_p_baseline,
#      agb.findActivity("treatment of hazardous waste, hazardous waste incineration", loc="RoW", db_name="ecoinvent-3.10-cutoff"): R_eou*INC*M_p_baseline,
#     agb.findActivity("treatment of municipal solid waste, sanitary landfill", loc="RoW", unit="kilogram", db_name="ecoinvent-3.10-cutoff"): R_eou*LF*M_p_baseline,
#   agb.findActivity("EI_manufacturing_baseline",db_name=DB_project_baseline):-0.5*R_eou*R_reuse, #product manufacturing impacts avoided by reuse
#    }
# )


# EI_eol_baseline_informal=agb.newActivity(DB_project_baseline, "EI_eol_baseline_informal", "unit", {
#       agb.findActivity("treatment of municipal solid waste, unsanitary landfill, dry infiltration class (100mm)", loc="GLO", db_name="ecoinvent-3.10-cutoff"): (1-R_eou)*WINF*0.5*M_p_baseline,
#       agb.findActivity("treatment of hazardous waste, hazardous waste incineration", loc="RoW", db_name="ecoinvent-3.10-cutoff"): (1-R_eou)*WINF*0.5*M_p_baseline,
#    }
# )

### Compile end of life impacts of baseline scenario

In [ ]:


# EI_eol_baseline = agb.newActivity(DB_project_baseline, "EI_eol_baseline", "unit", {
# EI_eol_baseline_informal:-(N_p_baseline+(1-r_repair)*RP),
# EI_eol_baseline_formal:-(N_p_baseline+(1-r_repair)*RP)
#     })

# Life Cycle Impacts Assessment

## Compile models from all life cycle stages

In [ ]:
# EI_total = agb.newActivity(DB_project_baseline, "EI_total", "unit", {
# EI_manufacturing_baseline:1,
# EI_transport_baseline:1,
# EI_use_baseline:1,
# EI_eol_baseline:1,
#     })

## Calculate all impacts categories for each life cycle stage

In [ ]:
# agb.compute_impacts({
# EI_manufacturing_baseline:1,
# EI_transport_baseline:1,
# EI_use_baseline:1,
# EI_eol_baseline:1,
#     }, all_impacts_categories)



## Calculate all impacts categories for the total life cycle stages

In [ ]:
# agb.compute_impacts({
# EI_total :1,
#     }, all_impacts_categories)

**Function for exporting the data to excel**

In [ ]:
# df.to_excel('filename.xlsx', index=False)

# LCIA Result interpretation

**Install the plotly package for visualisation**

In [ ]:
#pip install plotly

## Normalisation

### Define the normalisation factors

In [ ]:

# #EF Normalisation JRC report (Sala et al.,2017)

# factor_normalisation = [5.79E+13, 439000000, 4.5E+14, 3.83E+11, 8.15E+13, 266000, 3270000, 5060000000, 1.95E+11, 1.22E+12, 2.91E+13, 
#                        9.64E+15, 161000000, 4950000, 2.8E+11, 7.91E+13]

# name_normalisation = ['Climate change (GWP100)',	'Material resources: metals/minerals (ADP)', 'Energy resources: non-renewable (ADP)',
#                      'Acidification (AE)', 'Ecotoxicity: freshwater (CTUe)', 'Human toxicity: carcinogenic (CTUh)', 'Human toxicity: non-carcinogenic (CTUh)',
#                      'Eutrophication: freshwater (P)', 'Eutrophication: marine (N)', 'Eutrophication: terrestrial (AE)', 'Ionising radiation: human health', 
#                      'Land use', 'Ozone depletion (ODP)', 'Particulate matter formation', 'Photochemical oxidant formation: human health', 'Water use']


### Calculate normalised impacts

In [ ]:

# df = agb.compute_impacts(EI_total, all_impacts_categories)
# df = np.divide(df, factor_normalisation)
# df = df*100/sum(df.values[0])
# df.columns = name_normalisation
# autre = 0
# index = []
# for i in range(len(all_impacts_categories)):
#     if df[name_normalisation[i]].iloc[0]<1 and name_normalisation[i]!='<b>Climate change (GWP100)</b>':
#         autre = autre + df[name_normalisation[i]].iloc[0]
#         index.append(name_normalisation[i])
       
# for i in index:
#     df = df.drop(i, axis=1)
    
# df = df.T 
# df = df.sort_values(by = 'EI_total', ascending=False)
# df = df.T
# df[f'Other ({len(index)} indicators)']=[autre]

# # print(df)

### Visualisation

In [ ]:
# import plotly.graph_objects as go

# fig = go.Figure(go.Pie(
#     labels=df.columns,            # Labels for the pie slices
#     values=df.values[0],            # Values for the pie slices
#     name='Pie Chart',         # Name for the trace (used in legends)
#     pull=[0.05, 0.05, 0, 0, 0, 0, 0, 0.1, 0.05],         # Specify how much each slice should be pulled from the center (optional)
#     textposition='outside',
#     textinfo='percent+label', # Information to display on the pie slices
#     hoverinfo='label+percent',# Information to display on hover
#     #textfont=dict(style=['bold', 'bold', 'normal', 'normal', 'normal', 'normal', 'normal', 'bold', "normal"]),
#     sort=False,
#     hole = 0.35,
#     showlegend=False,
#     insidetextorientation='horizontal',
#     marker=dict(line=dict(color=['black', 'black', 'white', 'white', 'white', 'white', 'white', 'black', "white" ], width=[2, 2, 0, 0, 0, 0, 0, 2, 0]))  # Customize colors and borders
# ))
# fig.update_layout(
#     autosize=True,
#     width=950,
#     height=800,
#     title=go.layout.Title(text=f"Normalised impacts repartition of the DC-DC Buck converter based on Global Normalisation Factors for Environmental Footprint recommended by the European Commission"),
#     title_font_size=22,
#     title_automargin=False,  
#     annotations=[dict(text='Normalised impacts<br>(PEF methodology)', x=0.51, y=0.5, font_size=20, showarrow=False)]
# )


# fig.show()


## Distribution of target environmental impacts across life cycles 

Choose the target environmental impacts categories:
(example)
1. GWP100
2. ADP
3. CTUe

In [ ]:
# GWP100 = [climat[0]]
# df_GWP100_value = agb.compute_impacts({
# EI_manufacturing_baseline:1,
# EI_transport_baseline:1,
# EI_use_baseline:1,
# EI_eol_baseline:1,
#     }, GWP100)

# GWP100_total_value=sum(df_GWP100_value.values[:, 0])

# df_GWP100_percentage = df_GWP100_value*100/GWP100_total_value

# df_GWP100_percentage = df_GWP100_percentage.T


# # print(df_GWP100_value)
# # print(GWP100_total_value)
# # print(df_GWP100_percentage)
# # print(df_GWP100_percentage.values[0])

# import plotly.graph_objects as go

# fig = go.Figure(go.Pie(
#     labels=df_GWP100_percentage.columns,            # Labels for the pie slices
#     values=df_GWP100_percentage.values[0],            # Values for the pie slices
#     name='Pie Chart',         # Name for the trace (used in legends)
#     pull=[0, 0, 0, 0],         # Specify how much each slice should be pulled from the center (optional)
#     textinfo='percent', # Information to display on the pie slices
#     hoverinfo='label+percent',# Information to display on hover
#     #textfont=dict(style=['bold', 'bold', 'normal', 'normal', 'normal', 'normal', 'normal', 'bold', "normal"]),
#     sort=False,
#     hole = 0.40,
#     showlegend=True,
#     insidetextorientation='horizontal',
#     marker=dict(line=dict(color=['white', 'white', 'white', "white" ], width=[2, 1, 1, 1]))  # Customize colors and borders
# ))

# fig.update_layout(
#       autosize=True,
#       width=650,
#       height=500,
#       title=go.layout.Title(text=f"Distribution of GWP100/kg CO2 eq across life cycle stages"),
#       title_font_size=22,
#       title_automargin=False,  
#       annotations=[dict(text='GWP100/kg CO2 eq', x=0.51, y=0.5, font_size=12, showarrow=False)]
#   )



# fig.show()

In [ ]:

# ADP = [resources[0]]
# df_ADP_value = agb.compute_impacts({
# EI_manufacturing_baseline:1,
# EI_transport_baseline:1,
# EI_use_baseline:1,
# EI_eol_baseline:1,
#     }, ADP)

# ADP_total_value=sum(df_ADP_value.values[:, 0])
# df_ADP_percentage = df_ADP_value*100/sum(df_ADP_value.values[:, 0])
# df_ADP_percentage = df_ADP_percentage.T




# import plotly.graph_objects as go

# fig = go.Figure(go.Pie(
#     labels=df_ADP_percentage.columns,            # Labels for the pie slices
#     values=df_ADP_percentage.values[0],            # Values for the pie slices
#     name='Pie Chart',         # Name for the trace (used in legends)
#     pull=[0, 0, 0, 0],         # Specify how much each slice should be pulled from the center (optional)
#     textinfo='percent', # Information to display on the pie slices
#     hoverinfo='label+percent',# Information to display on hover
#     #textfont=dict(style=['bold', 'bold', 'normal', 'normal', 'normal', 'normal', 'normal', 'bold', "normal"]),
#     sort=False,
#     hole = 0.40,
#     showlegend=True,
#     insidetextorientation='horizontal',
#     marker=dict(line=dict(color=['white', 'white', 'white', "white" ], width=[2, 1, 1, 1]))  # Customize colors and borders
# ))

# fig.update_layout(
#       autosize=True,
#       width=650,
#       height=500,
#       title=go.layout.Title(text=f"ADP/kg Sb eq", x=0  # Anchor the title at the center
# ),
#       title_font_size=22,
#       title_automargin=False,  
#       annotations=[dict(text='ADP/kg Sb eq', x=0.51, y=0.5, font_size=12, showarrow=False)]
#   )


# fig.show()

In [ ]:

# CTUe = [ecotox[0]]
# df_CTUe_value = agb.compute_impacts({
# EI_manufacturing_baseline:1,
# EI_transport_baseline:1,
# EI_use_baseline:1,
# EI_eol_baseline:1,
#     }, CTUe)

# CTUe_total_value=sum(df_CTUe_value.values[:, 0])
# df_CTUe_percentage = df_CTUe_value*100/sum(df_CTUe_value.values[:, 0])

# df_CTUe_percentage = df_CTUe_percentage.T


# # print(df_CTUe_value)
# # print(CTUe_total_value)
# # print(df_CTUe_percentage)
# # print(df_CTUe_percentage.values[0])

# import plotly.graph_objects as go

# fig = go.Figure(go.Pie(
#     labels=df_CTUe_percentage.columns,            # Labels for the pie slices
#     values=df_CTUe_percentage.values[0],            # Values for the pie slices
#     name='Pie Chart',         # Name for the trace (used in legends)
#     pull=[0, 0, 0, 0],         # Specify how much each slice should be pulled from the center (optional)
#     textinfo='percent', # Information to display on the pie slices
#     hoverinfo='label+percent',# Information to display on hover
#     #textfont=dict(style=['bold', 'bold', 'normal', 'normal', 'normal', 'normal', 'normal', 'bold', "normal"]),
#     sort=False,
#     hole = 0.40,
#     showlegend=True,
#     insidetextorientation='horizontal',
#     marker=dict(line=dict(color=['white', 'white', 'white', "white" ], width=[2, 1, 1, 1]))  # Customize colors and borders
# ))

# fig.update_layout(
#       autosize=True,
#       width=650,
#       height=500,
#       title=go.layout.Title(text=f"CTUe", x=0  # Anchor the title at the center
# ),
#       title_font_size=22,
#       title_automargin=False,  
#       annotations=[dict(text='CTUe', x=0.51, y=0.5, font_size=12, showarrow=False)]
#   )



# fig.show()

## Distribution of manufacturing impacts across technical blocks

### Define list of target impacts categories

In [ ]:
# # List of impacts to consider
# climat = agb.findMethods("climate change", mainCat="EF v3.0")
# resources = agb.findMethods("material resources: metals/minerals", mainCat="EF v3.0")
# ecotox = agb.findMethods("ecotoxicity: freshwater", mainCat="EF v3.0")
# impacts = [climat[0], resources[0], ecotox[0]]
# #resources
# #climat
# impacts

### Calculation & Visualisation

In [ ]:
#  import plotly.graph_objects as go

# #Define the items to calculate and their name on the graph
# technical_blocks = {technicle_block_1:1,
#     technicle_block_2:1,
#     technicle_block_i:1,


# }

# technical_blocks_name = ["technicle_block_1", "technicle_block_2", "technicle_block_i"]

# colorphase = ["royalblue", "tomato", "orange"]



# unity = ["kgCO2 eq", "kgSb eq", "CTU eq"]
# indicators = ["climate change - global warming potential (GWP100)[kg CO2-Eq]","material resources: metals/minerals - abiotic depletion potential (ADP): elements (ultimate reserves)[kg Sb-Eq]","ecotoxicity: freshwater - comparative toxic unit for ecosystems (CTUe)[CTUe]"]
# indicator_simplified = ["Climate change (GWP100)","Abiotic depletion potential (ADP)","Ecotoxicity : freshwater (CTUe)"]

# total = agb.compute_impacts(
# EI_manufacturing_baseline
#     , impacts)                   
    

# df = agb.compute_impacts(technical_blocks, impacts)


# df = agb.compute_impacts(technical_blocks,impacts)/total.values 
# df.index = technical_blocks_name
# df['moyenne'] = pd.Series(df.mean(axis=1), index = technical_blocks_name)
# df = df.sort_values(by = 'moyenne', ascending = False)

# fig = go.Figure(data=[
#     go.Bar(name=indicator_simplified[0], x=df.index, y=df[indicators[0]], marker=dict(color = colorphase[0])),
#     go.Bar(name=indicator_simplified[1], x=df.index, y=df[indicators[1]], marker=dict(color = colorphase[1])),
#     go.Bar(name=indicator_simplified[2], x=df.index, y=df[indicators[2]], marker=dict(color = colorphase[2]))
# ])
# fig.update_layout(
#     autosize=True,
#     width=1100,
#     height=800,
    
#     title=go.layout.Title(text=f"Normalised impacts of buck converter manufacturing subparts, in 3 categories."),
#     title_font_size=22,
#     title_automargin=False,   
#     title_x=0.5,
#     yaxis_tickformat = '.0%'
# )
# fig.update_yaxes(title=f"Normalised impacts (%)")
# fig.update_xaxes(title=f"Buck converter subpart")
# fig.show()

#  Ecodesign Scenarios modelling

## Create scenarios 

Create scenario as switch parameters

In [ ]:
# design_scenarios = agb.newEnumParam(
#     'design_scenarios', 
#     label="design_scenarios",
#     values =[
#         "baseline",
#         "scenario1",
#         "scenario2",
#         "scenarioi",
#         "Combined_scenario_A",
#         "Combined_scenario_B",],
#     default="baseline",  
#     description="design scenarios")


## Define modified parameters 

For each scenario, redefine the concerned parameters.

Some parametrisation techniques are presented below:
1. If "number of technical block" changes:

In [ ]:
#Nb_technical_block_i=agb.switchValue(design_scenarios, baseline=1, refuse=0, rethink=1,reduce=1,Combined_scenario_A=0,Combined_scenario_B=0,repair=0,reuse=0) #existance of load's resistors

# EI_manufacturing_scenario =agb.newActivity(DB_project_baseline,
#         "EI_manufacturing_scenario", 
#         "unit",{
#     Technicl_block_i:Nb_technical_block_i*N_p,
#     ...
#         },)


2. If a "new technical block" is added

In [ ]:
# new_technical_block=agb.newActivity(DB_project_baseline,"new_technical_block","unit",{
#     agb.findActivity("component's reference", loc="GLO", db_name="ecoinvent-3.10-cutoff"):size of the component,
# })


# Nb_new_technical_block=agb.switchValue(design_scenarios, baseline=1, refuse=1, rethink=0,reduce=1,Combined_scenario_A=0,Combined_scenario_B=0,repair=0,reuse=0) #existance of Al plate


# EI_manufacturing_scenario =agb.newActivity(DB_project_baseline,
#         "EI_manufacturing_scenario", 
#         "unit",{
#     new_technicl_block_i:Nb_new_technical_block*N_p,
#     ...
#         },)

3. If "the size of one technical block" changes

In [ ]:
# M_TB_reduce = agb.newFloatParam(
#     'M_TB_reduce', 
#     label="proportion of environmental impacts of technical block i in reduce scenario compare to baseline",
#     default=1, min=0.5, max=2,   # Fixed if no min /max provided
#     distrib=agb.DistributionType.LINEAR,
#     description="proportion of environmental impacts of technical block i in reduce scenario compare to baseline")

# EI_manufacturing_scenario =agb.newActivity(DB_project_baseline,
#         "EI_manufacturing_scenario", 
#         "unit",{
#     technicl_block_i:M_TB_reduce*N_p
#     ...
#         },)

4. If "one component's size or technology in one technical block" changes

In [ ]:
#If you know the exactly data of the modified size of component

# component_i = agb.newSwitchAct(DB_project_baseline, 
#     "component_i", # Name
#     design_scenarios, # Sith parameter
#     { # Dictionnary of enum values / activities
#         "baseline" : (agb.findActivity("component's reference", db_name="ecoinvent-3.10-cutoff"),
#                 initial size of component),
#         "scenario" : (agb.findActivity("component's reference", db_name="ecoinvent-3.10-cutoff"),
#                 modified size of component),

#     })


#If you know the data range of the modified size of component
# M_EC_reduce = agb.newFloatParam(
#     'M_EC_reduce', 
#     label="proportion of environmental impacts of component i in reduce scenario compare to baseline",
#     default=1, min=0.5, max=2,   # Fixed if no min /max provided
#     distrib=agb.DistributionType.LINEAR,
#     description="proportion of environmental impacts of component i in reduce scenario compare to baseline")

# component_i = agb.newSwitchAct(DB_project_baseline, 
#     "component_i", # Name
#     design_scenarios, # Sith parameter
#     { # Dictionnary of enum values / activities
#         "baseline" : (agb.findActivity("component's reference", db_name="ecoinvent-3.10-cutoff"),
#                 initial size of component),
#         "scenario" : (agb.findActivity("component's reference", db_name="ecoinvent-3.10-cutoff"),
#                 M_EC_reduce*initial size of component),

#     })


**Start parameterisation!**

In [ ]:
# your turn!

## Compile Ecodesign scenarios' life cycle inventory 

### Manufacturing

In [ ]:
# EI_manufacturing_scenario =agb.newActivity(DB_project_baseline,
#         "EI_manufacturing_scenario", 
#         "unit",{
#     technical_block_1:1*N_p,
#     technical_block_2:1*N_p,
#     technical_block_i:1*N_p,
#     technical_block_i+1:1*N_p, #if there are new technical block added!
#     technical_block_i+n:1*N_p,
#         },)
# agb.printAct(EI_manufacturing_scenario)


### Transport

In [ ]:


# Transport_manufacturing_scenario=agb.newActivity(DB_project_baseline,
#         "Transport_manufacturing_scenario", 
#         "unit",{
#          agb.findActivity("market for transport, freight, sea, container ship", loc="GLO", db_name="ecoinvent-3.10-cutoff"):0.019*M_p*N_p,
#          agb.findActivity("market group for transport, freight, lorry, unspecified", loc="GLO", db_name="ecoinvent-3.10-cutoff"):0.001*M_p*N_p,
#         })

# Transport_eol_scenario =agb.newActivity(DB_project_baseline,
#         "Transport_eol_scenario", 
#         "unit",{
#          agb.findActivity("market group for transport, freight, lorry, unspecified", loc="GLO", db_name="ecoinvent-3.10-cutoff"):0.001*M_p*N_p*R_eou,
#          agb.findActivity("market for transport, freight, sea, container ship", loc="GLO", db_name="ecoinvent-3.10-cutoff"):0.019*M_p*N_p*WINF,
#          agb.findActivity("market group for transport, freight, lorry, unspecified", loc="GLO", db_name="ecoinvent-3.10-cutoff"):0.001*M_p*N_p*WINF,
#         })

# EI_transport_scenario = agb.newActivity(DB_project_baseline, "EI_transport_scenario", "unit", {
# Transport_manufacturing_scenario:1,
# Transport_eol_scenario:1
#     })

# agb.printAct(EI_transport_scenario)




### Use

In [ ]:
# EI_use=agb.newActivity(DB_project_baseline, "EI_use", "unit", {
#     agb.findActivity("market for electricity, low voltage", loc="FR", db_name="ecoinvent-3.10-cutoff"): P*T_op*(2-E),
#     })

# agb.printAct(EI_use)

### Repair

In [ ]:
# EI_repair=agb.newActivity(DB_project_baseline, "EI_repair", "unit", 
#                                          exchanges={
#                 agb.findActivity("EI_target_repair_components",db_name=DB_project_baseline):r_repair*RP,
#                 agb.findActivity("EI_manufacturing_scenario",db_name=DB_project_baseline):((1-r_repair)*RP) # end of life need to be added here
#                                          })    



### Reuse

In [ ]:

# EI_reuse =agb.newActivity(DB_project_baseline,
#         "EI_reuse", 
#         "unit",{
#                 agb.findActivity("EI_avoided_Target_reuse_components_manufacturing",db_name=DB_project_baseline)::-(N_p+(1-r_repair)*RP),
#        })



### End-of-life

In [ ]:
# EI_eol_scenario_formal=agb.newActivity(DB_project_baseline, "EI_eol_scenario_formal", "unit", {
#       agb.findActivity("treatment of waste electric and electronic equipment, shredding", loc="GLO", db_name="ecoinvent-3.10-cutoff"): R_eou*R_recycling*M_p,
#     agb.findActivity("treatment of used capacitor, to hazardous waste incineration, with energy recovery", unit="kilogram", loc="RoW", db_name="ecoinvent-3.10-cutoff"): R_eou*ER*M_p,
#      agb.findActivity("treatment of hazardous waste, hazardous waste incineration", loc="RoW", db_name="ecoinvent-3.10-cutoff"): R_eou*INC*M_p,
#     agb.findActivity("treatment of municipal solid waste, sanitary landfill", loc="RoW", unit="kilogram", db_name="ecoinvent-3.10-cutoff"): R_eou*LF*M_p,
#   agb.findActivity("EI_manufacturing_baseline",db_name=DB_project_baseline):-0.5*R_eou*R_reuse, #product manufacturing impacts avoided by reuse
#    }
# )


# EI_eol_scenario_informal=agb.newActivity(DB_project_baseline, "EI_eol_scenario_informal", "unit", {
#       agb.findActivity("treatment of municipal solid waste, unsanitary landfill, dry infiltration class (100mm)", loc="GLO", db_name="ecoinvent-3.10-cutoff"): (1-R_eou)*WINF*0.5*M_p,
#       agb.findActivity("treatment of hazardous waste, hazardous waste incineration", loc="RoW", db_name="ecoinvent-3.10-cutoff"): (1-R_eou)*WINF*0.5*M_p,
#    }
# )

# EI_eol_scenario = agb.newActivity(DB_project_baseline, "EI_eol_scenario", "unit", {
# EI_eol_scenario_informal:-(N_p+(1-r_repair)*RP),
# EI_eol_scenario_formal:-(N_p+(1-r_repair)*RP)
#     })

## Ecodesign scenarios evaluation

### Results of "BASELINE" scenario

In [ ]:
# df_baseline=agb.compute_impacts({
# EI_manufacturing_scenario:1,
# EI_transport_scenario:1,
# EI_use:1,
# EI_repair:0,
# EI_reuse:0,
# EI_eol_scenario:1,
#     }, impacts,design_scenarios="baseline")


### Results of scenario 1

In [ ]:
# df_scenario1=agb.compute_impacts({
# EI_manufacturing_scenario:1,
# EI_transport_scenario:1,
# EI_use:1,
# EI_repair:0,
# EI_reuse:0,
# EI_eol_scenario:1,
#     }, impacts,design_scenarios="scenario1")



### Results of combined scenario

In [ ]:
# df_combined=agb.compute_impacts({
# EI_manufacturing_scenario:1,
# EI_transport_scenario:1,
# EI_use:1,
# EI_repair:0,
# EI_reuse:0,
# EI_eol_scenario:1,
#     }, impacts,design_scenarios="scenario1")
